# Check Loading the VOC2012 Segmentation Dataset for Downstream Tasks

In [ ]:
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader

from oxels.datasets.voc2012_dataset import VOCDataset
# Define input image transformations
input_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define target (segmentation mask) transformations
# Using NEAREST interpolation to avoid smoothing label values
target_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.NEAREST),
    transforms.PILToTensor(),  # Produces 1 x H x W tensor with class indices
])

# Create the VOC2012 segmentation datasets

train_dataset = VOCDataset(
    year='2012',
    image_set='train',
    download=False,
    transform=input_transform,
    target_transform=target_transform
)

val_dataset = VOCDataset(
    year='2012',
    image_set='val',
    download=True,
    transform=input_transform,
    target_transform=target_transform
)

# Wrap datasets in DataLoaders
batch_size = 8
num_workers = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

# Check the dataset sizes
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

# VOC2012 Model

In [ ]:
import os
from oxels.models import ImageNetModel


backbone_run_name = "amber-terrain-77"
#backbone_run_name = "scarlet-sky-138"
backbone_run_name = "zesty-durian-7"
ckpt_dir = os.path.join("lightning_logs", str(backbone_run_name))
backbone_ckpt_path = os.path.join(ckpt_dir, backbone_run_name)
backbone = ImageNetModel.load_from_checkpoint(backbone_ckpt_path)
print(f"Loaded model from {os.path.join(ckpt_dir, 'best_model.ckpt')}")
num_oxels = backbone.hparams.num_oxels

# Some Pixel Classifier

In [ ]:
import torch
import torch.nn as nn
from torchmetrics.classification import Accuracy
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from oxels.models.metrics_mixin import MetricsMixin
import lightning as L


class Simple1x1Classifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1, head_groups: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.linear = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.linear(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x

class MLPStefanClassifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 150),
            nn.ReLU(),
            nn.Linear(150, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)  # [B, H, W, C]
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x


class DGLinearClassifier(MetricsMixin, L.LightningModule):
    def __init__(
        self,
        backbone: nn.Module,
        head: nn.Module,
        dataset_name: str,
        train_batch_size: int,
        val_batch_size: int,
        num_oxels: int = 64,
        image_size: int = 256,
        weight_decay: float = 0.004,
        learning_rate: float = 1e-3,
        lr_pct_start: float = 0.05,
        lr_div_factor: float = 25.0,
        lr_final_div_factor: float = 1e4,
    ):
        super().__init__()
        self.save_hyperparameters(
            ignore=["backbone", "head", "dataset_name"],
        )
        self.backbone = backbone
        self.backbone.freeze()
        self.head = head
        self.dataset_name = dataset_name
        self.loss_fn = nn.BCEWithLogitsLoss(reduction="mean")
        self.train_acc = Accuracy(task="binary", num_classes=head.num_classes)
        self.val_acc = Accuracy(task="binary", num_classes=head.num_classes)
        self.test_acc = Accuracy(task="binary", num_classes=head.num_classes)

    def forward(self, x):
        oxels = self.backbone(x)
        prediction = self.head(oxels)
        return prediction

    def compute_loss(self, batch):
        images, labels = batch
        logits = self(images) # some scalar
        B, _, H, W = logits.shape
        # turn labels into single value
        labels = torch.argmax(labels, dim=1)
        labels = labels.to(dtype=logits.dtype)
        labels = labels.view(B, 1, 1, 1).repeat(1, 1, H, W)  # Now (B, 1, H, W)
        loss = self.loss_fn(logits, labels)
        return loss

    def compute_metrics(self, batch):
        images, labels = batch
        logits = self(images)
        B, _, H, W = logits.shape
        #logits = logits.view(logits.shape[0], -1)  # Flatten the spatial dimensions
        labels = torch.argmax(labels, dim=1)
        labels = labels.to(dtype=logits.dtype)
        labels_pix = labels.view(B, 1, 1, 1).repeat(1, 1, H, W)  # Now (B, 1, H, W)
        loss = self.loss_fn(logits, labels_pix)
        # choose the right Accuracy object based on stage
        # Lightning will set self.training, self.validating, self.testing flags
        if self.trainer.training:
            logits = torch.sigmoid(logits)  # Apply sigmoid to logits for binary classification
            logits = torch.mean(logits, dim=[1, 2, 3])
            acc = self.train_acc(logits, labels)
        elif self.trainer.validating:
            logits = torch.sigmoid(logits)  # Apply sigmoid to logits for binary classification
            logits = torch.mean(logits, dim=[1, 2, 3])
            acc = self.val_acc(logits, labels)
        else:  # testing
            logits = torch.sigmoid(logits)  # Apply sigmoid to logits for binary classification
            logits = torch.mean(logits, dim=[1, 2, 3])
            acc = self.test_acc(logits, labels)

        return {
            "loss": loss,
            "accuracy": acc,
        }

    def configure_optimizers(self):
        lr = self.hparams.learning_rate
        wd = self.hparams.weight_decay

        # only use parameters that requires grad
        params = [p for p in self.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=lr, weight_decay=wd, betas=(0.9, 0.99))
        scheduler = OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=self.trainer.estimated_stepping_batches,
            div_factor=self.hparams.lr_div_factor,
            final_div_factor=self.hparams.lr_final_div_factor,
            pct_start=self.hparams.lr_pct_start,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        metrics = self.compute_metrics(batch)
        prefix = "validation/" if dataloader_idx == 0 else "validation/ood/"
        for key, value in metrics.items():
            key = f"{prefix}{key}"
            self.log(key,
                     value,
                     on_step=False,
                     on_epoch=True,
                     sync_dist=True,
                     prog_bar=(dataloader_idx == 0),  # maybe only show ID in prog bar
                     logger=True)

        return metrics["loss"]

    def train_dataloader(self):
        # Create the VOC2012 segmentation datasets
        train_dataset = VOCDataset(
            year='2012',
            image_set='train',
            download=False,
            transform=input_transform,
            target_transform=target_transform
        )
        return DataLoader(
            train_dataset,
            batch_size=self.hparams.train_batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=0,#len(os.sched_getaffinity(0)),
            drop_last=True,
        )

    def val_dataloader(self):

        val_dataset = VOCDataset(
            year='2012',
            image_set='val',
            download=True,
            transform=input_transform,
            target_transform=target_transform
        )


        return DataLoader(val_dataset, batch_size=self.hparams.val_batch_size, shuffle=False,  num_workers=0, drop_last=False)

